<a id="top"></a>

# Demo: prompt chaining -- your first workflow graph

```yaml
title: "Demo: prompt chaining -- your first workflow graph"
keywords: langgraph, StateGraph, prompt chaining, workflow, dag, node, edge, state, reducer, control flow as data, mixed models, haiku, sonnet, ChatAnthropic, Langfuse, teacher demo
estimated duration: 22
```

> **Lesson:** L04. Teacher demo -- Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L04/demos_or_activities.md).
> Makes **live** Claude calls -- set `ANTHROPIC_API_KEY` to run the workflow (the graph build and
> diagram run without a key). Mixes models per node: **Haiku 4.5** for the light `parse` step,
> **Sonnet 4.6** for `draft` and `policy_check`. Clear outputs before committing.

## Contents

- [1. Setup -- clients, tickets, tracing](#1-setup----clients-tickets-tracing)
- [2. The typed state](#2-the-typed-state)
- [3. The three nodes](#3-the-three-nodes)
- [4. Wire it, compile it, see the graph as data](#4-wire-it-compile-it-see-the-graph-as-data)
- [5. Run the workflow (live)](#5-run-the-workflow-live)
- [6. Read the trace](#6-read-the-trace-a-forward-pointing-taste-not-a-prerequisite)
- [7. Takeaways](#7-takeaways)

## 1. Setup -- clients, tickets, tracing

Three things, all self-contained: the **native LangChain `ChatAnthropic`** client (two of them --
a cheap Haiku and a capable Sonnet), a few sample **support tickets** plus a short **policy**, and a
`trace_callbacks()` helper that will route spans to Langfuse once it's configured -- the full
tracing platform is **L12's** lesson; here it's just a forward-pointing taste.

**Say the departure out loud:** continuing from [L03](../L03/objectives.md), nodes call
`ChatAnthropic` directly, *not* the `potato_llm` seam from L01-L02 -- "frameworks bring their own
client abstraction." The API key still loads through `common/config.py`; only the client changed.

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langgraph.graph import END, StateGraph

from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_anthropic_key,
)

SONNET = "claude-sonnet-4-6"  # course anchor -- the capable model for heavy reasoning nodes
HAIKU = "claude-haiku-4-5"  # the cheap, fast model for the light nodes (parse / classify / route)

# A handful of support tickets for the running example (the lesson's chosen domain).
TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
    "general": "Do you offer a student discount, and how would I apply it to my account?",
}

# A short policy the policy-check node holds a drafted reply against.
POLICY = (
    "Refunds: a duplicate charge is always refundable within 60 days. "
    "Never promise a refund for change-of-mind. "
    "Escalate anything mentioning legal action or chargebacks to a human."
)

# Live calls load the key through the config seam (common/config.py) -- same as L01-L02,
# only the *client* is the framework's now (continuing from L03). Build the clients only
# when a key is present so a keyless restart-and-run-all still passes; the run cells below
# are gated on LIVE.
LIVE = bool(get_settings().anthropic_api_key)
if LIVE:
    # Per-node model binding (objective 1): each node uses its OWN client. A node is an
    # independent call, so a graph can mix a cheap model and a capable one.
    haiku = ChatAnthropic(model=HAIKU, api_key=require_anthropic_key(), max_tokens=256)
    sonnet = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)


def trace_callbacks() -> list[object]:
    """Return LangChain callbacks that route spans to the shared Langfuse, or [].

    When Langfuse isn't configured the workflow runs unchanged -- only the spans are
    absent -- so this never blocks a run. L12 is where the Langfuse seam is TAUGHT in full;
    here we just get a forward-pointing taste of "watch the workflow run".
    """
    if not langfuse_configured():
        print("Langfuse not configured -- running without tracing. (L12 covers this in full.)")
        return []
    from langfuse.langchain import CallbackHandler

    return [CallbackHandler()]

[↑ Back to top](#top)

## 2. The typed state

The **state** is a typed object threaded through every node. It carries the data that flows
between steps. Note `steps`: its `add` **reducer** *appends* each node's name instead of
overwriting -- the same state/reducer machinery L11 reuses for an agent's message history.

In [ ]:
class TicketState(TypedDict):
    """State threaded through the parse -> draft -> policy_check chain.

    Example value after a full run:
        {"ticket":  "I was charged twice...",
         "parsed":  "Customer was double-charged and wants a refund.",
         "draft":   "Hi! So sorry about the duplicate charge...",
         "verdict": "OK: a duplicate charge is refundable within 60 days.",
         "steps":   ["parse", "draft", "policy_check"]}
    """

    ticket: str  # the raw incoming ticket (the input)
    parsed: str  # one-line issue summary the parse node extracts
    draft: str  # the reply the draft node writes
    verdict: str  # the policy-check node's ruling
    # `steps` ACCUMULATES: the `add` reducer appends, it does not overwrite. Every other
    # field uses the default reducer (last write wins).
    steps: Annotated[list[str], add]

[↑ Back to top](#top)

## 3. The three nodes

Each node is a plain typed function: it reads the state and returns a **partial update** -- just
the fields it changed -- never the whole state. The model lives *inside* these nodes; what runs
*next* is decided by the edges we wire in section 4, not by the model.

Watch the per-node model binding: `parse` uses **Haiku** (a light extraction), while `draft` and
`policy_check` use **Sonnet** (the reasoning steps).

In [ ]:
def parse(state: TicketState) -> dict[str, object]:
    """Summarize the ticket's core issue. A LIGHT step -> Haiku."""
    reply = haiku.invoke(
        f"In one sentence, summarize this support ticket's core issue:\n\n{state['ticket']}"
    )
    return {"parsed": str(reply.content), "steps": ["parse"]}


def draft(state: TicketState) -> dict[str, object]:
    """Write a customer reply from the parsed issue. The HEAVY step -> Sonnet."""
    reply = sonnet.invoke(
        f"Write a short, friendly support reply for this issue:\n\n{state['parsed']}"
    )
    return {"draft": str(reply.content), "steps": ["draft"]}


def policy_check(state: TicketState) -> dict[str, object]:
    """Check the draft against the policy snippet. A reasoning step -> Sonnet."""
    reply = sonnet.invoke(
        "You are a compliance reviewer. Reply with 'OK: <reason>' or 'REVISE: <reason>'.\n"
        f"Policy:\n{POLICY}\n\nDraft reply:\n{state['draft']}"
    )
    return {"verdict": str(reply.content), "steps": ["policy_check"]}

[↑ Back to top](#top)

## 4. Wire it, compile it, see the graph as data

The five-line `StateGraph` recipe: add the nodes, set the entry point, wire the **edges**, and
`compile()`. Then the payoff line of the whole lesson -- the compiled graph can **draw itself**.
That render needs no API key: the control flow *is* data you can inspect.

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("parse", parse)
builder.add_node("draft", draft)
builder.add_node("policy_check", policy_check)
builder.set_entry_point("parse")
builder.add_edge("parse", "draft")
builder.add_edge("draft", "policy_check")
builder.add_edge("policy_check", END)
chain_app = builder.compile()

# Control flow as DATA: every edge moves forward to END -- a DAG, no back-edge. That
# absence of a back-edge is exactly what makes this a *workflow*, not an agent.
print(chain_app.get_graph().draw_mermaid())

[↑ Back to top](#top)

## 5. Run the workflow (live)

Now `invoke()` it on one ticket, passing the Langfuse callback. Watch the `steps` field: the path
is **`['parse', 'draft', 'policy_check']` every time** -- the developer wired it, so it's
deterministic. (The *wording* of the draft varies run to run; the *path* does not.)

In [ ]:
if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live run. Set it to watch the chain execute.")
else:
    result = chain_app.invoke(
        {"ticket": TICKETS["billing"], "steps": []},
        config={"callbacks": trace_callbacks()},
    )
    print("path   :", result["steps"])  # always ['parse', 'draft', 'policy_check']
    print("\nparsed :", result["parsed"])
    print("\ndraft  :\n", result["draft"])
    print("\nverdict:", result["verdict"])

[↑ Back to top](#top)

## 6. Read the trace (a forward-pointing taste, not a prerequisite)

If Langfuse is configured, open the run there (the same self-hosted instance **L12** will teach in
full). A prompt-chaining run shows a **linear chain** of generation spans -- one per node, in
order: `parse` -> `draft` -> `policy_check`. That linear trace *confirms* the path was
developer-determined.

Look at the model on each span: `parse` ran on **Haiku**, `draft` and `policy_check` on **Sonnet**.
A light cost/latency aside -- the extraction step is cheap; the reasoning steps are where the spend
goes. (*Which* model each step deserves is **L14's** topic; here we just see *that* a graph lets
you mix.) **If Langfuse isn't configured the run still works** -- you just won't see the spans, and
that's fine: reading a full run as a structured trace is L12's job, not this lesson's.

[↑ Back to top](#top)

## 7. Takeaways

- **Control flow is now data.** Nodes and edges can be listed and drawn -- unlike `if`/`while`
  buried in Python. The diagram is the proof; the trace (L12) will be another.
- **The model lives inside the nodes; the developer owns the edges.** The model parses and drafts;
  *what runs next* is the edges you wired.
- **First framework, continued.** Since [L03](../L03/objectives.md), nodes call `ChatAnthropic`
  directly, not the `potato_llm` seam.
- **Why decompose into three prompts?** Smaller focused prompts are more reliable and individually
  testable; the cost is more calls (the L01 trade). For a strictly linear chain the graph is near
  break-even -- its payoff shows with branching (that's **L05**, next), shared state, and tracing
  (L12).

[↑ Back to top](#top)